# Prompt Quality Analyzer — training and evaluation notebook

**Goal:** train a lightweight model that distinguishes original prompts from their optimized counterparts in the BPO dataset, then export a reproducible artifact for the FastAPI app.

> **Important interpretation:** BPO contains original/optimized pairs, not objective human-assigned quality scores from 0–100. The model trained here estimates a *relative optimized-prompt preference signal* (`0 = original prompt`, `1 = optimized prompt`). It does **not** estimate a universal truth about prompt quality or guarantee a better answer from every LLM. The product combines this signal with a separate explainable rubric and discloses both.

### Dataset selected
- **BPO (Black-box Prompt Optimization):** https://huggingface.co/datasets/zai-org/BPO
- Research paper: https://arxiv.org/abs/2311.04155
- The Hugging Face dataset card currently identifies the license as Apache-2.0 and lists 14,395 prompt optimization pairs. The notebook verifies the downloaded record counts and available columns rather than hardcoding them as truth.

Run each cell from top to bottom in a fresh Google Colab runtime. The first install/download cells require internet access. The notebook uses the published train/validation/test splits and filters exact text overlap between splits before evaluation.

## Stage A — Environment and data loading

Only the libraries needed for data handling, visualization, TF-IDF modelling and artifact export are installed.

In [ ]:
%pip -q install "datasets>=3.0,<5" "scikit-learn>=1.5,<2" "pandas>=2.0,<4" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "joblib>=1.4,<2"

In [ ]:
from __future__ import annotations
import json, os, re, random, shutil, hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from datasets import load_dataset
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, precision_recall_fscore_support,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DATASET_ID = "zai-org/BPO"
DATASET_URL = "https://huggingface.co/datasets/zai-org/BPO"
PAPER_URL = "https://arxiv.org/abs/2311.04155"
DATASET_LICENSE_AS_LISTED = "Apache-2.0; verify current dataset card before redistribution"

print("Python environment ready")
print("Dataset:", DATASET_ID)
print("Random seed:", SEED)

In [ ]:
# Fetch using the official Hugging Face datasets loader and preserve its named splits.
raw_dataset = load_dataset(DATASET_ID)
print("Available splits:", list(raw_dataset.keys()))
print("Downloaded records by split:", {name: len(split) for name, split in raw_dataset.items()})
print("Available columns:", raw_dataset[ list(raw_dataset.keys())[0] ].column_names)

# The expected task fields are verified below; stop rather than silently train on the wrong schema.
required_columns = {"prompt", "optimized_prompt"}
missing_columns = required_columns - set(raw_dataset[list(raw_dataset.keys())[0]].column_names)
if missing_columns:
    raise ValueError(f"Expected prompt-pair columns not found: {sorted(missing_columns)}")
if "train" not in raw_dataset or "test" not in raw_dataset:
    raise ValueError("The dataset must provide train and test splits. Inspect the dataset card and update split mapping if it changes.")
VALID_SPLIT = "valid" if "valid" in raw_dataset else ("validation" if "validation" in raw_dataset else None)
if VALID_SPLIT is None:
    raise ValueError("No validation split found. Create a group-aware validation split before model selection.")

## Stage B — Exploratory data analysis

The original prompt and optimized prompt are two sides of a pair. For this task, their role is the label source; `good_res` and `bad_res` describe response preferences and are **not** used as prompt-quality scores.

In [ ]:
dataframes = {name: split.to_pandas() for name, split in raw_dataset.items()}
train_source = dataframes["train"].copy()
valid_source = dataframes[VALID_SPLIT].copy()
test_source = dataframes["test"].copy()

print("TRAIN shape:", train_source.shape)
print("TRAIN dtypes:")
print(train_source.dtypes)
print("\nMissing values in train:")
print(train_source.isna().sum().sort_values(ascending=False).head(12))
print("\nExact duplicate rows in train:", int(train_source.duplicated().sum()))
print("\nSample records (selected columns):")
columns_to_show = [column for column in ["prompt", "optimized_prompt", "good_res", "bad_res"] if column in train_source.columns]
display(train_source[columns_to_show].head(5))

In [ ]:
# Inspect whether optimized prompts tend to be longer in this dataset. This is an EDA observation,
# not proof that longer prompts are always better.
eda = train_source[["prompt", "optimized_prompt"]].copy()
eda["prompt"] = eda["prompt"].fillna("").astype(str)
eda["optimized_prompt"] = eda["optimized_prompt"].fillna("").astype(str)
eda["original_words"] = eda["prompt"].str.split().str.len()
eda["optimized_words"] = eda["optimized_prompt"].str.split().str.len()

plt.figure(figsize=(9, 4))
plt.hist(eda["original_words"].clip(upper=250), bins=40, alpha=.62, label="Original prompt")
plt.hist(eda["optimized_words"].clip(upper=250), bins=40, alpha=.62, label="Optimized prompt")
plt.title("Prompt length distribution (values above 250 words clipped for visibility)")
plt.xlabel("Word count")
plt.ylabel("Number of records")
plt.legend()
plt.tight_layout()
plt.show()

display(eda[["original_words", "optimized_words"]].describe().round(2))

## Stage C — Cleaning, labels and leakage checks

Each public record becomes a pair: original prompt → label `0`, optimized prompt → label `1`. We drop empty pairs and exact duplicate pairs. The official split boundaries are preserved. Evaluation pairs that share an exact normalized prompt string with training data are removed to reduce direct leakage. This does not detect every paraphrase or semantic near-duplicate, so the limitation is recorded.

In [ ]:
def normalize_for_overlap(text: str) -> str:
    text = str(text).casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def prepare_pairs(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    pair_frame = frame[["prompt", "optimized_prompt"]].copy()
    pair_frame["prompt"] = pair_frame["prompt"].fillna("").astype(str).str.strip()
    pair_frame["optimized_prompt"] = pair_frame["optimized_prompt"].fillna("").astype(str).str.strip()
    pair_frame = pair_frame[(pair_frame["prompt"] != "") & (pair_frame["optimized_prompt"] != "")].copy()
    pair_frame["_norm_original"] = pair_frame["prompt"].map(normalize_for_overlap)
    pair_frame["_norm_optimized"] = pair_frame["optimized_prompt"].map(normalize_for_overlap)
    pair_frame = pair_frame[pair_frame["_norm_original"] != pair_frame["_norm_optimized"]]
    pair_frame = pair_frame.drop_duplicates(subset=["_norm_original", "_norm_optimized"])
    pair_frame["pair_id"] = [f"{split_name}-{i:06d}" for i in range(len(pair_frame))]
    pair_frame["split"] = split_name
    return pair_frame.reset_index(drop=True)

train_pairs = prepare_pairs(train_source, "train")
valid_pairs = prepare_pairs(valid_source, "valid")
test_pairs = prepare_pairs(test_source, "test")

train_texts_for_overlap = set(train_pairs["_norm_original"]) | set(train_pairs["_norm_optimized"])

def remove_train_overlap(pairs: pd.DataFrame, split_name: str) -> tuple[pd.DataFrame, int]:
    overlap = pairs["_norm_original"].isin(train_texts_for_overlap) | pairs["_norm_optimized"].isin(train_texts_for_overlap)
    dropped = int(overlap.sum())
    return pairs.loc[~overlap].reset_index(drop=True), dropped

valid_pairs, valid_overlap_removed = remove_train_overlap(valid_pairs, "valid")
test_pairs, test_train_overlap_removed = remove_train_overlap(test_pairs, "test")
# Avoid test-to-validation overlap as well, so model selection cannot see exact test text.
valid_texts_for_overlap = set(valid_pairs["_norm_original"]) | set(valid_pairs["_norm_optimized"])
test_valid_overlap = test_pairs["_norm_original"].isin(valid_texts_for_overlap) | test_pairs["_norm_optimized"].isin(valid_texts_for_overlap)
test_valid_overlap_removed = int(test_valid_overlap.sum())
test_pairs = test_pairs.loc[~test_valid_overlap].reset_index(drop=True)

print("Clean pairs:", {"train": len(train_pairs), "valid": len(valid_pairs), "test": len(test_pairs)})
print("Validation pairs removed due to exact train-text overlap:", valid_overlap_removed)
print("Test pairs removed due to exact train-text overlap:", test_train_overlap_removed)
print("Test pairs removed due to exact validation-text overlap:", test_valid_overlap_removed)
if min(len(train_pairs), len(valid_pairs), len(test_pairs)) == 0:
    raise ValueError("At least one cleaned split is empty. Inspect overlap filtering and source splits before training.")

# Convert a pair to two prompt-level classification records without separating paired sides across splits.
def expand_pairs(pairs: pd.DataFrame) -> pd.DataFrame:
    records = []
    for row in pairs.itertuples(index=False):
        records.append({"text": row.prompt, "label": 0, "pair_id": row.pair_id, "split": row.split, "role": "original"})
        records.append({"text": row.optimized_prompt, "label": 1, "pair_id": row.pair_id, "split": row.split, "role": "optimized"})
    return pd.DataFrame(records)

train_data = expand_pairs(train_pairs)
valid_data = expand_pairs(valid_pairs)
test_data = expand_pairs(test_pairs)
print("Prompt-level label counts (train):")
display(train_data["label"].map({0: "original (0)", 1: "optimized (1)"}).value_counts().rename("records").to_frame())

## Stage D — Baseline and stronger lightweight model

We compare (1) word unigram/bigram TF-IDF and (2) combined word plus character TF-IDF with logistic regression. We choose using **validation F1** only. The held-out test split is used once after model selection. Since class labels represent original-versus-optimized pair position rather than expert scores, all metrics evaluate that narrow proxy task.

In [ ]:
def build_model(use_char_features: bool) -> Pipeline:
    word = TfidfVectorizer(ngram_range=(1, 2), max_features=30000, min_df=1, sublinear_tf=True, strip_accents="unicode")
    transformers = [("word", word)]
    if use_char_features:
        char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=30000, min_df=1, sublinear_tf=True, strip_accents="unicode")
        transformers.append(("char", char))
    features = FeatureUnion(transformers)
    classifier = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    return Pipeline([("features", features), ("classifier", classifier)])


def classification_metrics(y_true, y_pred, probabilities) -> dict:
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    try:
        metrics["roc_auc"] = roc_auc_score(y_true, probabilities)
    except ValueError:
        metrics["roc_auc"] = float("nan")
    return {key: round(float(value), 4) for key, value in metrics.items()}

X_train, y_train = train_data["text"].tolist(), train_data["label"].to_numpy()
X_valid, y_valid = valid_data["text"].tolist(), valid_data["label"].to_numpy()
X_test, y_test = test_data["text"].tolist(), test_data["label"].to_numpy()

baseline_model = build_model(use_char_features=False)
combined_model = build_model(use_char_features=True)
validation_results = []

for model_name, model in [("word_tfidf_baseline", baseline_model), ("word_plus_char_tfidf", combined_model)]:
    model.fit(X_train, y_train)
    predicted = model.predict(X_valid)
    probability = model.predict_proba(X_valid)[:, 1]
    metrics = classification_metrics(y_valid, predicted, probability)
    validation_results.append({"model": model_name, **metrics})

validation_table = pd.DataFrame(validation_results).sort_values(["f1", "roc_auc"], ascending=False).reset_index(drop=True)
display(validation_table)
selected_name = str(validation_table.loc[0, "model"])
selected_model = baseline_model if selected_name == "word_tfidf_baseline" else combined_model
print("Selected by validation F1, then ROC-AUC:", selected_name)

In [ ]:
# Refit the selected model using training + validation records after model selection.
# The test set remains untouched until the next cell.
combined_train_validation = pd.concat([train_data, valid_data], ignore_index=True)
selected_model.fit(combined_train_validation["text"].tolist(), combined_train_validation["label"].to_numpy())

test_pred = selected_model.predict(X_test)
test_prob = selected_model.predict_proba(X_test)[:, 1]
test_metrics = classification_metrics(y_test, test_pred, test_prob)
print("Final held-out test metrics:")
print(json.dumps(test_metrics, indent=2))
print("\nClass-wise report:")
print(classification_report(y_test, test_pred, target_names=["original-style prompt", "optimized-style prompt"], zero_division=0))
print("Confusion matrix (rows=true labels 0/1, columns=predicted labels 0/1):")
print(confusion_matrix(y_test, test_pred))

## Stage E/F — Error analysis, complexity slices and robustness probes

These are diagnostic checks, not a guarantee of invariance. A style classifier can be sensitive to wording and length. Use these observations to decide whether more human-labeled evaluation data is needed.

In [ ]:
error_frame = test_data.copy()
error_frame["predicted_label"] = test_pred
error_frame["optimized_probability"] = test_prob
error_frame["correct"] = error_frame["label"] == error_frame["predicted_label"]
print("Correct vs incorrect predictions:")
display(error_frame["correct"].value_counts().rename(index={True: "correct", False: "incorrect"}).to_frame("records"))
print("Examples of false positives and false negatives:")
errors = error_frame[~error_frame["correct"]].sort_values("optimized_probability")
display(errors[["text", "role", "label", "predicted_label", "optimized_probability"]].head(12))

In [ ]:
# Broad complexity buckets are based only on prompt word count; these are not semantic difficulty labels.
error_frame["word_count"] = error_frame["text"].str.split().str.len()
error_frame["complexity_bucket"] = pd.cut(
    error_frame["word_count"], bins=[-1, 12, 40, np.inf], labels=["simple (0-12 words)", "medium (13-40 words)", "complex (>40 words)"]
)
complexity_rows = []
for bucket, group in error_frame.groupby("complexity_bucket", observed=False):
    if len(group) == 0:
        continue
    complexity_rows.append({"bucket": str(bucket), "records": len(group), "accuracy": round(float((group["label"] == group["predicted_label"]).mean()), 4), "f1": round(float(f1_score(group["label"], group["predicted_label"], zero_division=0)), 4)})
display(pd.DataFrame(complexity_rows))

In [ ]:
robustness_examples = [
    "Explain recursion to a beginner with one example.",
    "Explain recursion to a beginner with one example. Please be comprehensive, detailed, thorough, and expansive in every possible respect.",
    "Task: Explain recursion to a beginner. Output: one example and a short explanation.",
    "Could you explain recursion simply, and include an example for a beginner?",
]
robustness_table = pd.DataFrame({
    "prompt": robustness_examples,
    "optimized_style_probability": selected_model.predict_proba(robustness_examples)[:, 1],
})
display(robustness_table)
print("Interpretation: large score changes from filler or paraphrases would be evidence to improve the model or add human-labelled tests, not something to hide.")

## Stage G — Save versioned artifacts and metadata

The saved object includes vectorizers and the classifier, so the backend can load a single file without retraining during requests. A model card and metrics are written beside it. The artifact is a pair-preference proxy; independent human-reviewed prompt-quality labels are still required before presenting its output as validated quality prediction.

In [ ]:
ARTIFACT_DIR = Path("/content/model_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "prompt_preference.joblib"
METADATA_PATH = ARTIFACT_DIR / "metadata.json"
PREDICTIONS_PATH = ARTIFACT_DIR / "test_predictions.csv"

joblib.dump(selected_model, MODEL_PATH, compress=3)

metadata = {
    "model_version": "pqa-bpo-pair-preference-v1.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_type": "word_tfidf_logistic_regression" if selected_name == "word_tfidf_baseline" else "word_and_character_tfidf_logistic_regression",
    "training_data": "zai-org/BPO (Black-box Prompt Optimization), original-versus-optimized prompt pairs",
    "dataset_url": DATASET_URL,
    "research_paper_url": PAPER_URL,
    "dataset_license_as_listed": DATASET_LICENSE_AS_LISTED,
    "source_records": {name: len(split) for name, split in raw_dataset.items()},
    "clean_pairs": {"train": len(train_pairs), "valid": len(valid_pairs), "test": len(test_pairs)},
    "evaluation_overlap_pairs_removed": {"valid_vs_train": valid_overlap_removed, "test_vs_train": test_train_overlap_removed, "test_vs_valid": test_valid_overlap_removed},
    "label_semantics": "0=original prompt, 1=optimized prompt from a dataset pair; relative optimized-style proxy, not universal prompt-quality ground truth",
    "feature_config": {"word_ngrams": [1, 2], "word_max_features": 30000, "char_features_used": selected_name == "word_plus_char_tfidf", "char_ngrams": [3, 5] if selected_name == "word_plus_char_tfidf" else None, "char_max_features": 30000 if selected_name == "word_plus_char_tfidf" else None},
    "random_seed": SEED,
    "selection_metric": "validation F1, ROC-AUC used as tie-breaker",
    "validation_comparison": validation_results,
    "metrics": {"test": test_metrics},
    "limitations": [
        "BPO contains prompt optimization pairs, not expert-scored universal 0-100 prompt-quality labels.",
        "Optimized prompts in this dataset were generated in a research pipeline and may contain imperfect edits.",
        "English-only; broad task diversity does not cover every language or domain.",
        "The learned probability is an optimized-style preference signal, not a calibrated probability of good LLM output.",
        "Exact normalized prompt overlaps with training text were removed from validation and test; semantic near-duplicates may remain.",
        "Prompt length and formatting cues may act as shortcuts. Inspect robustness probes and add a human-reviewed evaluation set before production claims.",
    ],
    "production_ready": False,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
error_frame.to_csv(PREDICTIONS_PATH, index=False)
print("Saved:", MODEL_PATH, "bytes:", MODEL_PATH.stat().st_size)
print("Saved:", METADATA_PATH)
print(json.dumps(metadata, indent=2))

In [ ]:
# Clean-runtime smoke test: load the artifact from disk and predict two unseen prompts.
reloaded_model = joblib.load(MODEL_PATH)
smoke_prompts = [
    "help me with my resume",
    "Please review my resume for a software internship. Prioritize project evidence, skills relevance, and clarity. Return five specific edits and do not invent experience.",
]
smoke_probabilities = reloaded_model.predict_proba(smoke_prompts)[:, 1]
display(pd.DataFrame({"prompt": smoke_prompts, "optimized_style_probability": smoke_probabilities}))
assert len(smoke_probabilities) == 2
assert np.isfinite(smoke_probabilities).all()
assert np.all((smoke_probabilities >= 0) & (smoke_probabilities <= 1))

# Also load the artifact in a fresh Python subprocess, which catches serialization/import issues
# that a notebook variable already in memory could hide.
import subprocess, sys
subprocess_script = r"""
import json, sys, joblib, numpy as np
model = joblib.load(sys.argv[1])
prompts = ["help me with my resume", "Review my resume for a software internship. Return five specific edits."]
probabilities = model.predict_proba(prompts)[:, 1]
assert len(probabilities) == 2 and np.isfinite(probabilities).all()
assert np.all((probabilities >= 0) & (probabilities <= 1))
print(json.dumps([round(float(value), 4) for value in probabilities]))
"""
subprocess_result = subprocess.run(
    [sys.executable, "-c", subprocess_script, str(MODEL_PATH)],
    capture_output=True, text=True, check=True,
)
print("Fresh-process artifact check:", subprocess_result.stdout.strip())
print("Artifact reload and prediction checks passed in this runtime.")

## Export the artifact to your computer

Download the zip and copy its two runtime files into the repository's `backend/models/` folder. Keep the accompanying metadata; it documents the label semantics and evaluation limits. This notebook has **not** been executed in this conversation's environment: its successful output can only be established after you run all cells in Colab.

In [ ]:
ZIP_PATH = shutil.make_archive("/content/prompt_quality_model_artifacts", "zip", root_dir=ARTIFACT_DIR)
print("Artifact zip ready:", ZIP_PATH)
try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception:
    print("Not running in Google Colab; retrieve the artifact zip from:", ZIP_PATH)

## What the exported model means — and does not mean

- It was trained on prompt-pair examples from BPO, and its test metrics describe the task of predicting whether a text is the optimized side of a known pair.
- The product's overall score blends 80% transparent heuristic rubric with 20% learned pair-preference signal. This mix is a design choice, not a scientifically established weighting.
- Do not call the result “80% accurate”, do not treat the probability as a validated quality score, and do not claim higher scores guarantee better real-world answers.
- Before a serious release, collect human-rated prompts across tasks, add an untouched independent test set, assess inter-rater agreement and score calibration, and test whether superficial length/format changes alter predictions unfairly.
- The rewrite fallback uses placeholders and task-aware templates. If you enable a hosted rewrite provider, obtain consent in the UI and keep its key on the backend.